# 004 · Processes with multiprocessing

> **Standard Library › Concurrency** · Python 3.14

`multiprocessing` runs Python code in **separate operating-system processes**, each with its own interpreter and its own GIL. It is the standard way to use all CPU cores for pure-Python CPU-bound work. Its API deliberately mirrors `threading` (`Process` ≈ `Thread`), plus process pools and tools for passing data between processes.

**In this notebook you will learn:**

- starting and joining a `Process`, exit codes
- `Pool`: `map`, `imap_unordered`, `starmap`, `apply_async`
- start methods (`spawn`, `forkserver`, `fork`) and why worker functions must be importable — especially in notebooks
- the `if __name__ == "__main__":` guard in scripts
- sharing data: `Queue`, `Pipe`, `Value`/`Array`, `Manager`
- the costs: process start-up and pickling

## .NET mental model

There is **no direct .NET equivalent**, because .NET doesn't need one: the CLR has no GIL, so `Task.Run`, `Parallel.For` and PLINQ already run CPU-bound C# code on all cores using **threads** in a single process.

| Python `multiprocessing` | Closest .NET idea |
|---|---|
| `Process(target=f).start()` | `Process.Start("worker.exe")` — but running a *function*, not an exe |
| `Pool().map(f, items)` | `Parallel.ForEach` / PLINQ (which use threads) |
| `Queue`, `Pipe` | anonymous/named pipes, `System.IO.Pipes` |
| `Value`, `Array` (shared memory) | `MemoryMappedFile` |
| `Manager().dict()` | a small out-of-process state server (remoting, gRPC) |
| pickling arguments/results | serialising messages between processes |

So think of `multiprocessing` as "a convenience layer for spawning worker processes and talking to them", and reach for it only for CPU-bound Python code. For I/O-bound work, threads or `asyncio` are simpler and cheaper.

**Tip:** for most "run this function over these inputs" jobs, `concurrent.futures.ProcessPoolExecutor` is the friendlier API on top of the same machinery *(see Standard Library › Concurrency › 003 · Executors with concurrent.futures)*. Use `multiprocessing` directly when you need `Process` objects, queues, pipes or shared state.

## Start methods — and why functions must be importable

A child process needs to know which function to run. How it gets there depends on the **start method**:

| Method | How the child starts | Default on |
|---|---|---|
| `spawn` | a fresh Python interpreter that **imports** your main module, then unpickles the function by name | macOS, Windows |
| `forkserver` | children are forked from a clean server process; the function is still pickled by name | Linux and other POSIX (since 3.14) |
| `fork` | a copy of the parent process (inherits everything; unsafe with threads) | Linux before 3.14 |

With `spawn` and `forkserver`, the target function is pickled **by reference** (module name + function name) and the child imports it. A function defined in a notebook cell lives in the kernel's `__main__`, which a child process cannot import, so it fails with `AttributeError: module '__main__' has no attribute '...'` — and with `Pool.map` the workers may even keep failing silently and **hang**.

In [ ]:
import multiprocessing as mp

print(mp.get_start_method())   # 'spawn' on macOS/Windows, 'forkserver' on Linux (3.14+)

In a real project, worker functions simply live in a `.py` module. To stay self-contained, this notebook writes such a module into a temporary directory and puts that directory on `sys.path` (children started with `spawn` receive the parent's `sys.path`):

In [ ]:
import sys
import tempfile
import textwrap
from pathlib import Path

tmp = tempfile.TemporaryDirectory()
Path(tmp.name, "mp_workers.py").write_text(textwrap.dedent("""
    import os
    import time

    def square(n):
        return n * n

    def power(base, exp):
        return base ** exp

    def slow_square(n):
        time.sleep(0.02 * (n % 3))      # finish in a scrambled order
        return n * n

    def report(q, name):
        q.put((name, os.getpid()))

    def fail():
        raise ValueError("boom")

    def produce(q, n):
        for i in range(n):
            q.put(i * i)
        q.put(None)                      # sentinel: no more data

    def shout_server(conn):
        while (msg := conn.recv()) is not None:
            conn.send(msg.upper())
        conn.close()

    def add_many(counter, times):
        for _ in range(times):
            with counter.get_lock():
                counter.value += 1

    def fill(arr, index):
        arr[index] = index * 10

    def register(shared, name):
        shared[name] = len(name)
"""))
sys.path.insert(0, tmp.name)

import mp_workers
print("module ready:", mp_workers.square(7))

## `Process` — start, join, exit code

```csharp
// C# — the nearest thing: run another program
using var p = Process.Start("worker.exe", "--job 1");
p.WaitForExit();
Console.WriteLine(p.ExitCode);
```

`multiprocessing.Process` has the same shape as `threading.Thread`: `target`, `args`, `start()`, `join()`, `is_alive()`, `daemon` — plus `pid` and `exitcode`.

In [ ]:
import os

q = mp.Queue()
procs = [mp.Process(target=mp_workers.report, args=(q, f"worker-{i}")) for i in range(3)]
for p in procs:
    p.start()
results = [q.get() for _ in procs]     # read results BEFORE joining (see gotcha below)
for p in procs:
    p.join()

print(sorted(name for name, _ in results))
print("all ran in other processes:", all(pid != os.getpid() for _, pid in results))
print("exit codes:", [p.exitcode for p in procs])   # 0 = success

An exception in the child doesn't propagate to the parent either — the child prints a traceback to *its* stderr and exits with code `1`:

In [ ]:
p = mp.Process(target=mp_workers.fail)
p.start()
p.join()
print("exitcode:", p.exitcode)          # 1 → the child raised

The `Process Process-N: Traceback ...` text above is the **child's** stderr, not an exception in this cell.

⚠️ **Gotcha:** a child process writes `print()` output and tracebacks straight to the operating-system stdout/stderr it inherited. Current Jupyter kernels capture those and show them (as above), but depending on the environment the output may instead appear only in the terminal that launched Jupyter, be attached to a later cell, or interleave unpredictably between processes. Never rely on printing from children for results — send them back through a `Queue`, `Pipe` or a pool's return values.

## `Pool` — a pool of worker processes

```csharp
// C#
var squares = numbers.AsParallel().AsOrdered().Select(Square).ToArray();
```

`Pool(processes=n)` starts `n` workers (default: number of CPUs). Its methods:

- `map(fn, items)` — results in input order (blocks until all done)
- `imap` / `imap_unordered(fn, items)` — lazy iterators; `imap_unordered` yields results as they finish
- `starmap(fn, [(a, b), ...])` — unpacks argument tuples
- `apply_async(fn, args)` — submit one call, get an `AsyncResult` (`.get()` ≈ `future.result()`)

In [ ]:
with mp.Pool(processes=4) as pool:
    print(pool.map(mp_workers.square, range(8)))

    unordered = list(pool.imap_unordered(mp_workers.slow_square, range(6)))
    print("imap_unordered (sorted):", sorted(unordered))   # arrival order varies

    print(pool.starmap(mp_workers.power, [(2, 3), (3, 2), (10, 0)]))

    async_result = pool.apply_async(mp_workers.square, (12,))
    print(async_result.get(timeout=5))

⚠️ **Gotcha:** leaving a `with Pool() as pool:` block calls `pool.terminate()`, **not** `close()` + `join()` — any still-running work is killed. Make sure you've collected all results (`map` blocks; for `imap`/`apply_async` consume them) before the block ends.

**Tip:** for many tiny tasks pass `chunksize=` to `map`/`imap` so each worker receives items in batches rather than one pickle round-trip per item.

## The `if __name__ == "__main__":` guard (scripts)

With `spawn` (and `forkserver`), each child **imports your main module**. If that module creates processes at import time, every child would try to create its own children. So in a script you must put the "launch" code behind the main guard:

```python
# crunch.py  — run with: uv run python crunch.py
from multiprocessing import Pool

def work(n):                 # module-level → importable by the children
    return n * n

if __name__ == "__main__":   # only the parent process runs this
    with Pool() as pool:
        print(pool.map(work, range(10)))
```

Without the guard you get a `RuntimeError` explaining that "an attempt has been made to start a new process before the current process has finished its bootstrapping phase". In a notebook, the kernel itself is the main process, so no guard is needed — but the functions must still come from an importable module.

## Sharing data between processes

Processes don't share memory: each has its own copy of every object. A change in a child is invisible to the parent. You must communicate explicitly.

### `Queue` — message passing

`multiprocessing.Queue` has the same API as `queue.Queue` (`put`, `get`, `get(timeout=...)`), but items are pickled and sent through a pipe. It is the process-level `Channel<T>`.

In [ ]:
q = mp.Queue()
producer = mp.Process(target=mp_workers.produce, args=(q, 5))
producer.start()

received = []
while (item := q.get(timeout=5)) is not None:   # stop at the sentinel
    received.append(item)
producer.join()
print(received)

⚠️ **Gotcha:** a process that has put items on a `Queue` won't terminate until those items have been flushed to the pipe. If the parent calls `join()` **before** draining the queue, both can wait for each other forever. Rule: **get first, join after** (as above).

### `Pipe` — a two-way connection between two processes

`Pipe()` returns two `Connection` objects; whatever one end `send()`s, the other `recv()`s — like a pair of connected `NamedPipeServerStream`/`NamedPipeClientStream`.

In [ ]:
parent_end, child_end = mp.Pipe()
server = mp.Process(target=mp_workers.shout_server, args=(child_end,))
server.start()

for word in ["hello", "pipes"]:
    parent_end.send(word)
    print(parent_end.recv())
parent_end.send(None)          # tell the child to stop
server.join()
parent_end.close()

### `Value` and `Array` — shared memory

`Value(typecode, initial)` and `Array(typecode, size)` allocate primitive values in **shared memory** (typecodes as in the `array` module: `'i'` int, `'d'` double). They come with a lock; increments are read-modify-write, so take it with `get_lock()` — the process-level version of the race condition from *Standard Library › Concurrency › 002 · Locks, Events and Queues*.

In [ ]:
counter = mp.Value("i", 0)
workers = [mp.Process(target=mp_workers.add_many, args=(counter, 500)) for _ in range(3)]
for w in workers: w.start()
for w in workers: w.join()
print("counter:", counter.value)       # 1500

arr = mp.Array("i", 4)                 # four ints, zero-initialised
fillers = [mp.Process(target=mp_workers.fill, args=(arr, i)) for i in range(4)]
for f in fillers: f.start()
for f in fillers: f.join()
print("array:", list(arr))

### `Manager` — shared Python objects via a server process

A `Manager()` starts a helper process that owns real Python objects (`dict`, `list`, `Namespace`, `Lock`, ...) and hands out **proxies**. Every operation on a proxy is a message to the manager — flexible, but much slower than `Value`/`Array`.

In [ ]:
with mp.Manager() as manager:
    shared = manager.dict()
    ps = [mp.Process(target=mp_workers.register, args=(shared, name)) for name in ["alice", "bob", "carol"]]
    for p in ps: p.start()
    for p in ps: p.join()
    print(sorted(shared.items()))     # copy the data out before the manager shuts down

**Tip:** prefer passing data in and returning results out (Pool/executor style) over shared state. It's simpler, and it's how you'd design it across machines anyway.

## The costs: start-up and pickling

Processes are heavy compared to threads:

1. **Start-up** — `spawn` launches a new interpreter and imports modules: tens of milliseconds per process.
2. **Pickling** — every argument and result is serialised, sent through a pipe and deserialised.

So multiprocessing only pays off when each task does substantially more work than it costs to ship. Watch a pool lose against a plain loop on trivial tasks:

In [ ]:
import time

start = time.perf_counter()
p = mp.Process(target=mp_workers.square, args=(1,))
p.start(); p.join()
print(f"one process start + join: ~{(time.perf_counter() - start) * 1000:.0f} ms")

numbers = list(range(20_000))
start = time.perf_counter()
plain = [mp_workers.square(n) for n in numbers]
t_plain = time.perf_counter() - start

with mp.Pool(4) as pool:
    start = time.perf_counter()
    pooled = pool.map(mp_workers.square, numbers)
    t_pool = time.perf_counter() - start

print("same results:", plain == pooled)
print(f"plain loop faster for trivial work: {t_plain < t_pool}")

And the pickled size of the data you send is real bytes over a pipe:

In [ ]:
import pickle

payload = list(range(100_000))
print(f"pickled list of 100k ints: {len(pickle.dumps(payload)) / 1024:.0f} KiB")

⚠️ **Gotcha:** objects that can't be pickled — lambdas, nested functions, open files, sockets, locks from `threading`, database connections — can't be sent to another process. Create such resources **inside** the worker (e.g. with a `Pool(initializer=...)`) instead.

Clean up the temporary module:

In [ ]:
sys.path.remove(tmp.name)
del sys.modules["mp_workers"]
tmp.cleanup()
print("cleaned up")

## Summary

| Task | C# / .NET | Python |
|---|---|---|
| Parallel CPU work | `Parallel.For`, PLINQ (threads) | `Pool.map` / `ProcessPoolExecutor` |
| Start a worker | `Process.Start(exe)` | `mp.Process(target=f, args=...).start()` |
| Wait / exit code | `WaitForExit()` / `ExitCode` | `join()` / `exitcode` |
| Ordered parallel map | `AsParallel().AsOrdered()` | `pool.map(f, xs)` |
| Results as they finish | — | `pool.imap_unordered(f, xs)` |
| Multiple args | — | `pool.starmap(f, [(a, b), ...])` |
| Message passing | pipes | `mp.Queue`, `mp.Pipe` |
| Shared primitives | `MemoryMappedFile` | `mp.Value`, `mp.Array` |
| Shared dict/list | — | `mp.Manager().dict()` |
| Script entry point | `Main` | `if __name__ == "__main__":` guard |

## Exercises

These exercises need worker functions in an importable module. The solutions write one into a temporary directory, like the notebook does.

1. Write a module function `count_primes(limit)` (naive trial division is fine) and use `Pool.map` to compute it for `[1000, 2000, 3000, 4000]`. Print the results.
2. Use `Pool.starmap` with a module function `describe(name, qty)` returning `f"{qty} x {name}"` over `[("apple", 3), ("pear", 5)]`.
3. Start 2 processes that each add 100 to a shared `mp.Value("i", 0)` one unit at a time (using `get_lock()`), and print the final value (200).

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
import multiprocessing as mp
import sys, tempfile, textwrap
from pathlib import Path

tmp = tempfile.TemporaryDirectory()
Path(tmp.name, "ex_workers.py").write_text(textwrap.dedent("""
    def count_primes(limit):
        return sum(
            1 for n in range(2, limit)
            if all(n % d for d in range(2, int(n ** 0.5) + 1))
        )

    def describe(name, qty):
        return f"{qty} x {name}"

    def add_units(counter, times):
        for _ in range(times):
            with counter.get_lock():
                counter.value += 1
"""))
sys.path.insert(0, tmp.name)
import ex_workers

# 1 — Pool.map keeps input order
with mp.Pool(4) as pool:
    print(pool.map(ex_workers.count_primes, [1000, 2000, 3000, 4000]))

    # 2 — starmap unpacks tuples
    print(pool.starmap(ex_workers.describe, [("apple", 3), ("pear", 5)]))

# 3 — shared Value with its lock
counter = mp.Value("i", 0)
ps = [mp.Process(target=ex_workers.add_units, args=(counter, 100)) for _ in range(2)]
for p in ps: p.start()
for p in ps: p.join()
print(counter.value)

sys.path.remove(tmp.name)
tmp.cleanup()
```

</details>